# Notebook 11 — Posterior Predictive Checks: Model Criticism

## Background

A model that sampled without divergences, with good R-hat and ESS, is not necessarily a *correct* model — it's a model that NUTS was able to sample from efficiently. The sampling was successful, but the model might still be wrong in important ways. Posterior predictive checks (PPCs) are how you find out.

The core idea is simple: if the model is correct, data simulated from the model's posterior predictive distribution should look like the actual data. If simulated data systematically looks different — in its mean, variance, tail behavior, or any other feature — the model is misspecified.

This is George Box's famous insight operationalized: "All models are wrong, but some are useful." PPCs tell you *how* the model is wrong and *whether* that matters for your purpose.

## What You'll Learn

- The PPC workflow: simulate -> compare -> diagnose
- Test statistics: what to compare beyond the marginal distribution
- Bayesian p-values: when are they meaningful?
- Graphical PPCs: ArviZ tools for visual model criticism
- Common misspecifications and their PPC signatures
- The LOO-PIT (leave-one-out probability integral transform): a rigorous calibration check
- Model improvement cycle: iterate until PPCs pass

## Why This Matters

Without PPCs, you might present results from a model that systematically under-predicts variance, misses temporal autocorrelation, or fails to handle zeros in count data. PPCs are the difference between fitting a model and actually understanding what it can and can't explain. They're the Bayesian version of residual analysis — and arguably more informative.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — The PPC Workflow

The PPC workflow has three steps:
1. **Simulate**: draw replicated datasets `y_rep` from `P(y|data)` — the posterior predictive
2. **Compare**: compute test statistics on both `y_obs` and each `y_rep`
3. **Diagnose**: if the observed statistic is extreme relative to the replicated distribution, the model fails that test

The **Bayesian p-value** `p_B = P(T(y_rep) >= T(y_obs))` measures how extreme the observed statistic is. Values near 0 or 1 indicate failure. Values near 0.5 indicate the model successfully replicates that statistic.

Unlike frequentist p-values, Bayesian p-values near 0.5 are *good* — they mean the model fits.

In [ ]:
np.random.seed(42)

# Dataset: overdispersed counts (insurance claims per customer)
# True generating process: Negative Binomial
true_mu = 3.5
true_alpha = 2.0  # low alpha = high overdispersion
n_obs = 200

# NegBin via Gamma-Poisson mixture
rates = np.random.gamma(true_alpha, true_mu / true_alpha, n_obs)
claims = np.random.poisson(rates)

print(f'Claims dataset: n={n_obs}')
print(f'  Mean:     {claims.mean():.2f} (true: {true_mu})')
print(f'  Variance: {claims.var():.2f} (Poisson would predict: {claims.mean():.2f})')
print(f'  Var/Mean: {claims.var()/claims.mean():.2f} (>1 = overdispersed)')

# Fit Model 1: Poisson (misspecified -- ignores overdispersion)
with pm.Model() as poisson_model:
    mu = pm.HalfNormal('mu', sigma=10)
    y_obs = pm.Poisson('y_obs', mu=mu, observed=claims)
    trace_pois = pm.sample(draws=1000, tune=500, chains=2,
                            random_seed=42, progressbar=False)
    ppc_pois   = pm.sample_posterior_predictive(trace_pois, random_seed=42)

# Fit Model 2: Negative Binomial (correct specification)
with pm.Model() as negbin_model:
    mu    = pm.HalfNormal('mu', sigma=10)
    alpha = pm.Exponential('alpha', lam=1)
    y_obs = pm.NegativeBinomial('y_obs', mu=mu, alpha=alpha, observed=claims)
    trace_nb = pm.sample(draws=1000, tune=500, chains=4,
                          random_seed=42, progressbar=False)
    ppc_nb   = pm.sample_posterior_predictive(trace_nb, random_seed=42)

print(f'\nPoisson mu: {trace_pois.posterior["mu"].values.mean():.2f}')
print(f'NegBin mu:  {trace_nb.posterior["mu"].values.mean():.2f}')
print(f'NegBin alpha: {trace_nb.posterior["alpha"].values.mean():.2f} (true: {true_alpha})')

In [ ]:
# Compute Bayesian p-values for multiple test statistics
def bayesian_pvalue(y_obs, y_rep, statistic_fn):
    """Fraction of replicated datasets where the test statistic exceeds the observed."""
    obs_val  = statistic_fn(y_obs)
    rep_vals = np.array([statistic_fn(y_rep[i]) for i in range(len(y_rep))])
    return obs_val, rep_vals, np.mean(rep_vals >= obs_val)

test_stats = {
    'Mean':       np.mean,
    'Variance':   np.var,
    'Var/Mean':   lambda y: np.var(y) / (np.mean(y) + 1e-8),
    'P(y=0)':     lambda y: np.mean(y == 0),
    '95th pct':   lambda y: np.percentile(y, 95),
    'Max':        np.max,
}

print('Bayesian p-values (good = close to 0.5, bad = near 0 or 1):')
print(f'{"Statistic":15} {"Observed":10} {"Pois p_B":10} {"NB p_B":10}')
print('-' * 48)

y_rep_pois = ppc_pois.posterior_predictive['y_obs'].values.reshape(-1, n_obs)
y_rep_nb   = ppc_nb.posterior_predictive['y_obs'].values.reshape(-1, n_obs)

for stat_name, stat_fn in test_stats.items():
    obs_val, rp, p_pois = bayesian_pvalue(claims, y_rep_pois, stat_fn)
    _,       _,  p_nb   = bayesian_pvalue(claims, y_rep_nb,   stat_fn)
    flag_pois = '  <- FAIL' if abs(p_pois - 0.5) > 0.4 else ''
    flag_nb   = '  <- FAIL' if abs(p_nb   - 0.5) > 0.4 else ''
    print(f'{stat_name:15} {obs_val:10.3f} {p_pois:10.3f}{flag_pois}   {p_nb:10.3f}{flag_nb}')

In [ ]:
# Visualize PPCs for both models
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

models_ppc = [
    ('Poisson (misspecified)', y_rep_pois, 'darkorange'),
    ('Negative Binomial (correct)', y_rep_nb, 'steelblue'),
]

for row, (model_name, y_rep, color) in enumerate(models_ppc):
    stats_to_plot = [
        ('Variance', np.var),
        ('Var/Mean ratio', lambda y: np.var(y)/(np.mean(y)+1e-8)),
        ('95th percentile', lambda y: np.percentile(y, 95)),
    ]
    
    for col, (stat_name, stat_fn) in enumerate(stats_to_plot):
        ax = axes[row, col]
        obs_val, rep_vals, p_b = bayesian_pvalue(claims, y_rep, stat_fn)
        
        ax.hist(rep_vals, bins=40, color=color, alpha=0.7, edgecolor='white', density=True)
        ax.axvline(obs_val, color='red', lw=2, linestyle='--',
                   label=f'Observed: {obs_val:.2f}')
        ax.set_title(f'{model_name}\n{stat_name}: p_B = {p_b:.3f}', fontsize=9)
        ax.set_xlabel(stat_name)
        ax.legend(fontsize=7)
        if abs(p_b - 0.5) > 0.4:
            ax.patch.set_facecolor('#FFEEEE')

plt.suptitle('PPC Comparison: Poisson vs Negative Binomial\n'
             'Red background = test statistic FAILS', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ArviZ built-in PPC visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (ppc, title) in zip(axes, [
    (ppc_pois, 'Poisson'),
    (ppc_nb,   'Negative Binomial'),
]):
    az.plot_ppc(ppc, observed=True, num_pp_samples=100, ax=ax,
                observed_rug=True)
    ax.set_title(f'{title}: Posterior Predictive\nBlack=observed KDE, Gray=replicated')

plt.tight_layout()
plt.show()

## Part 2 — Diagnosing Common Misspecifications

Different types of model misspecification have characteristic PPC signatures. Knowing the patterns helps you identify what to fix.

In [ ]:
# Case study: time series with autocorrelation -- Normal regression fails
np.random.seed(42)

n_ts = 100
t = np.arange(n_ts)
rho_ar = 0.7  # true AR(1) coefficient

# AR(1) process: y_t = 0.7 * y_{t-1} + epsilon
y_ts = np.zeros(n_ts)
for i in range(1, n_ts):
    y_ts[i] = rho_ar * y_ts[i-1] + np.random.normal(0, 1)
y_ts += 0.05 * t  # add a linear trend

# Fit a simple linear regression (ignores autocorrelation)
t_z = (t - t.mean()) / t.std()

with pm.Model() as linear_ts_model:
    alpha = pm.Normal('alpha', mu=0, sigma=5)
    beta  = pm.Normal('beta',  mu=0, sigma=3)
    sigma = pm.HalfNormal('sigma', sigma=2)
    mu_ts = alpha + beta * t_z
    y_obs = pm.Normal('y_obs', mu=mu_ts, sigma=sigma, observed=y_ts)
    trace_ts = pm.sample(draws=1000, tune=500, chains=2,
                          random_seed=42, progressbar=False)
    ppc_ts   = pm.sample_posterior_predictive(trace_ts, random_seed=42)

y_rep_ts = ppc_ts.posterior_predictive['y_obs'].values.reshape(-1, n_ts)

# Test statistic that detects autocorrelation
def autocorrelation_lag1(y):
    """Pearson correlation between y[t] and y[t-1]."""
    return np.corrcoef(y[:-1], y[1:])[0, 1]

obs_acf1, rep_acf1, p_b_acf1 = bayesian_pvalue(y_ts, y_rep_ts, autocorrelation_lag1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Replicated ACF(1) vs observed
ax = axes[0]
ax.hist(rep_acf1, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(obs_acf1, color='red', lw=2, linestyle='--',
           label=f'Observed ACF(1): {obs_acf1:.3f}')
ax.set_xlabel('Lag-1 autocorrelation'); ax.set_ylabel('Density')
ax.set_title(f'PPC: Autocorrelation Test\nBayesian p = {p_b_acf1:.4f} <- FAIL')
ax.legend()

# Time series with replicated paths
ax = axes[1]
for i in range(30):
    ax.plot(t, y_rep_ts[i], alpha=0.05, color='steelblue', lw=0.5)
ax.plot(t, y_ts, 'black', lw=2, label='Observed')
ax.set_xlabel('Time'); ax.set_ylabel('y')
ax.set_title('PPC: Time Series\nReplicated data (blue) lacks autocorrelation structure')
ax.legend()

plt.suptitle('Linear Regression Fails on AR(1) Data: PPC Detects It', fontsize=12)
plt.tight_layout()
plt.show()

print(f'True lag-1 autocorrelation: {rho_ar:.2f}')
print(f'Observed: {obs_acf1:.3f}')
print(f'Bayesian p-value: {p_b_acf1:.4f} -- nearly 0, strong evidence of misspecification')
print('Fix: use an AR(1) likelihood or a GP with Matern kernel')

In [ ]:
# PPC signature gallery: common misspecification patterns
np.random.seed(42)

n_sig = 200

# Scenario 1: Zero inflation -- Poisson misses excess zeros
# ZIP: 40% structural zeros + 60% Poisson(3)
structural_zero = np.random.binomial(1, 0.4, n_sig)
y_zip = np.where(structural_zero == 1, 0, np.random.poisson(3, n_sig))

# Scenario 2: Skewed continuous -- Normal misses skew
y_skew = np.random.lognormal(1, 0.5, n_sig)  # right-skewed

# Fit Poisson to ZIP data
with pm.Model() as poisson_zip:
    mu = pm.HalfNormal('mu', sigma=5)
    y  = pm.Poisson('y', mu=mu, observed=y_zip)
    tr = pm.sample(draws=500, tune=300, chains=2, random_seed=42, progressbar=False)
    pp = pm.sample_posterior_predictive(tr, random_seed=42)

# Fit Normal to lognormal data
with pm.Model() as normal_skew:
    mu  = pm.Normal('mu', mu=0, sigma=10)
    sig = pm.HalfNormal('sigma', sigma=5)
    y   = pm.Normal('y', mu=mu, sigma=sig, observed=y_skew)
    tr2 = pm.sample(draws=500, tune=300, chains=2, random_seed=42, progressbar=False)
    pp2 = pm.sample_posterior_predictive(tr2, random_seed=42)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# ZIP scenario
y_rep_zip = pp.posterior_predictive['y'].values.reshape(-1, n_sig)
obs_p0, rep_p0_vals, p_b_p0 = bayesian_pvalue(y_zip, y_rep_zip, lambda y: np.mean(y==0))

ax = axes[0, 0]
max_count = min(y_zip.max(), 15)
bins_zip = np.arange(-0.5, max_count+1.5)
for i in range(50):
    h, _ = np.histogram(y_rep_zip[i], bins=bins_zip, density=True)
    ax.step(bins_zip[:-1]+0.5, h, where='mid', alpha=0.05, color='steelblue')
h_obs, _ = np.histogram(y_zip, bins=bins_zip, density=True)
ax.step(bins_zip[:-1]+0.5, h_obs, where='mid', color='black', lw=2, label='Observed')
ax.set_title(f'Zero-Inflated Data + Poisson Model\nP(Y=0) p_B={p_b_p0:.3f} <- FAIL')
ax.set_xlabel('Count'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.hist(rep_p0_vals, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(obs_p0, color='red', lw=2, linestyle='--',
           label=f'Observed P(Y=0): {obs_p0:.3f}')
ax.set_xlabel('P(Y=0)'); ax.set_title('PPC: Zero-proportion test')
ax.legend(fontsize=8)

# Skewed data
y_rep_skew = pp2.posterior_predictive['y'].values.reshape(-1, n_sig)
obs_skew, rep_skew_vals, p_b_skew = bayesian_pvalue(y_skew, y_rep_skew,
                                                       lambda y: stats.skew(y))

ax = axes[1, 0]
az.plot_ppc({'posterior_predictive': {'y': pp2.posterior_predictive['y']}},
             num_pp_samples=80, ax=ax)
ax.set_title(f'Skewed Data + Normal Model\nPPC distribution: wide mismatch')

ax = axes[1, 1]
ax.hist(rep_skew_vals, bins=40, color='darkorange', alpha=0.7, edgecolor='white', density=True)
ax.axvline(obs_skew, color='red', lw=2, linestyle='--',
           label=f'Observed skew: {obs_skew:.2f}')
ax.set_xlabel('Skewness'); ax.set_title(f'PPC: Skewness test (p_B={p_b_skew:.4f})')
ax.legend(fontsize=8)

plt.suptitle('PPC Signature Gallery: Common Misspecification Patterns', fontsize=12)
plt.tight_layout()
plt.show()

print('Misspecification -> PPC signature:')
print('  Overdispersion: replicated variance << observed variance')
print('  Zero inflation: replicated P(Y=0) << observed P(Y=0)')
print('  Skewed data:    replicated skewness ~ 0 << observed skewness')
print('  Autocorrelation: replicated ACF(1) ~ 0 << observed ACF(1)')

## Part 3 — LOO-PIT: Calibration Check

The **LOO Probability Integral Transform** (PIT) checks whether the leave-one-out predictive distributions are calibrated. If the model is correct, `PIT_i = P(y_rep < y_i | data_{-i})` should be uniformly distributed on [0, 1].

- **Uniform PIT**: model is calibrated
- **U-shaped PIT**: model has too-narrow predictive intervals (overconfident)
- **Hump-shaped PIT**: model has too-wide intervals (underconfident)
- **Skewed PIT**: model is biased

In [ ]:
# LOO-PIT via ArviZ
# Compare correctly specified vs misspecified model

# Reuse the claims data: Poisson vs NegBin
with poisson_model:
    pm.compute_log_likelihood(trace_pois)
with negbin_model:
    pm.compute_log_likelihood(trace_nb)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (trace, ppc_obj, title) in zip(axes, [
    (trace_pois, ppc_pois, 'Poisson (misspecified)'),
    (trace_nb,   ppc_nb,   'Negative Binomial (correct)'),
]):
    az.plot_loo_pit(idata=trace, y='y_obs',
                    y_hat=ppc_obj.posterior_predictive['y_obs'],
                    ax=ax)
    ax.set_title(f'{title}\nLOO-PIT: uniform = calibrated')

plt.suptitle('LOO-PIT Calibration Check\n'
             'U-shaped = too narrow (overconfident), Uniform = well-calibrated', fontsize=12)
plt.tight_layout()
plt.show()

## Part 4 — The Model Improvement Cycle

PPCs are not just for detecting failure — they guide model improvement. The workflow is iterative:

1. Fit a baseline model
2. Run PPCs: which test statistics fail?
3. Hypothesize: what structural feature could cause this failure?
4. Modify the model to address it
5. Re-run PPCs: did the fix work? Did it introduce new failures?
6. Compare models with LOO-CV: does the improvement in fit justify complexity?

In [ ]:
# Full improvement cycle: fixing a zero-inflated model

np.random.seed(42)
n_zip = 150
# True: 35% structural zeros; 65% Poisson(4)
is_zero = np.random.binomial(1, 0.35, n_zip)
y_zip2  = np.where(is_zero == 1, 0, np.random.poisson(4, n_zip))

print(f'Zero-inflated dataset: n={n_zip}')
print(f'  Observed P(Y=0): {np.mean(y_zip2==0):.2f} (true structural zero rate: 0.35)')
print(f'  Mean: {y_zip2.mean():.2f}, Variance: {y_zip2.var():.2f}')

# Step 1: Baseline Poisson
with pm.Model() as baseline_poisson:
    mu = pm.HalfNormal('mu', sigma=5)
    y  = pm.Poisson('y', mu=mu, observed=y_zip2)
    tr_base = pm.sample(draws=1000, tune=500, chains=2, random_seed=42, progressbar=False)
    pp_base  = pm.sample_posterior_predictive(tr_base, random_seed=42)
    pm.compute_log_likelihood(tr_base)

# Step 2: Identify failure
y_rep_base = pp_base.posterior_predictive['y'].values.reshape(-1, n_zip)
_, _, p_b_zero_base = bayesian_pvalue(y_zip2, y_rep_base, lambda y: np.mean(y==0))
print(f'\nBaseline Poisson: P(Y=0) Bayesian p = {p_b_zero_base:.4f} <- FAIL')

# Step 3: Fix with Zero-Inflated Poisson
with pm.Model() as zip_model:
    psi = pm.Beta('psi', alpha=1, beta=1)  # probability of structural zero
    mu  = pm.HalfNormal('mu', sigma=5)
    y   = pm.ZeroInflatedPoisson('y', psi=psi, mu=mu, observed=y_zip2)
    tr_zip  = pm.sample(draws=1000, tune=500, chains=4,
                          random_seed=42, progressbar=False)
    pp_zip  = pm.sample_posterior_predictive(tr_zip, random_seed=42)
    pm.compute_log_likelihood(tr_zip)

y_rep_zip2 = pp_zip.posterior_predictive['y'].values.reshape(-1, n_zip)
_, _, p_b_zero_zip = bayesian_pvalue(y_zip2, y_rep_zip2, lambda y: np.mean(y==0))

psi_samp = tr_zip.posterior['psi'].values.flatten()
mu_samp  = tr_zip.posterior['mu'].values.flatten()

print(f'ZIP model: P(Y=0) Bayesian p = {p_b_zero_zip:.4f}  <- PASS')
print(f'ZIP psi (structural zero rate): {psi_samp.mean():.3f} (94% CI: {np.percentile(psi_samp, 3):.3f}, {np.percentile(psi_samp, 97):.3f})')
print(f'ZIP mu (Poisson rate): {mu_samp.mean():.3f} (true: 4.0)')

In [ ]:
# Compare baseline vs ZIP model with LOO-CV
comparison_zip = az.compare(
    {'Poisson': tr_base, 'ZeroInflatedPoisson': tr_zip},
    ic='loo', scale='log'
)
print('LOO-CV: Poisson vs Zero-Inflated Poisson')
print(comparison_zip.to_string())

az.plot_compare(comparison_zip, figsize=(8, 3))
plt.title('LOO-CV: Baseline Poisson vs Zero-Inflated Poisson')
plt.tight_layout()
plt.show()

# Visual PPC comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (y_rep_plot, title) in zip(axes, [
    (y_rep_base, 'Baseline Poisson (fails P(Y=0))'),
    (y_rep_zip2, 'Zero-Inflated Poisson (passes)'),
]):
    max_c = min(y_zip2.max(), 20)
    bins  = np.arange(-0.5, max_c + 1.5)
    for i in range(50):
        h, _ = np.histogram(y_rep_plot[i], bins=bins, density=True)
        ax.step(bins[:-1]+0.5, h, where='mid', alpha=0.05, color='steelblue')
    h_obs, _ = np.histogram(y_zip2, bins=bins, density=True)
    ax.step(bins[:-1]+0.5, h_obs, where='mid', color='black', lw=2, label='Observed')
    ax.set_title(title); ax.set_xlabel('Count'); ax.legend(fontsize=8)

plt.suptitle('PPC Before and After Model Fix', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

**PPC Workflow**:
```python
with model:
    trace = pm.sample(...)
    ppc = pm.sample_posterior_predictive(trace)

# ArviZ plot
az.plot_ppc(ppc, observed=True, num_pp_samples=100)

# Bayesian p-value for specific test statistic
y_rep = ppc.posterior_predictive['y_obs'].values.reshape(-1, n)
p_B = np.mean([statistic(y_rep[i]) >= statistic(y_obs) for i in range(len(y_rep))])
# Good: p_B near 0.5. Fail: near 0 or 1.

# LOO-PIT calibration check
with model:
    pm.compute_log_likelihood(trace)
az.plot_loo_pit(idata=trace, y='y_obs', y_hat=ppc.posterior_predictive['y_obs'])
```

**Test statistic selection**:
- Always check: mean, variance, tails (5th, 95th percentiles)
- Domain-specific: P(Y=0) for counts, ACF(1) for time series, skewness for skewed data
- Residual-like: per-observation predicted vs actual deviations

**Common misspecifications**:
| Problem | PPC signature | Fix |
|---------|--------------|-----|
| Overdispersion | Var p_B ~ 0 | NegBin likelihood |
| Zero inflation | P(Y=0) p_B ~ 0 | ZeroInflatedPoisson |
| Heavy tails | Tail p_B ~ 0 | Student-t likelihood |
| Autocorrelation | ACF p_B ~ 0 | AR(1) model or GP |
| Skew | Skewness p_B ~ 0 | Lognormal or Gamma likelihood |

**Next**: Notebook 12 — ROPE & Decision Theory. How do you make decisions from posterior distributions? The region of practical equivalence, loss functions, and decision-theoretic Bayesian inference.